# 🤖 Auto-Browser: Free Model Fine-Tuning Pipeline
### Fine-Tune Qwen-2.5-1.5B on Recruiter Screening Q&A using LoRA on Free Google Colab T4 GPU

This notebook trains a specialized AI agent on the candidate application dataset exported from **Auto-Browser** (), and saves the trained model for **FREE** to **Hugging Face Hub** or **Google Drive**.

## Step 1: Install Dependencies (1 minute)
We use Hugging Face ,  (LoRA),  (SFTTrainer), and  (4-bit quantization).

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub
!nvidia-smi

## Step 2: Clone Auto-Browser Repository or Upload Dataset
Clone your repository or upload .

In [ ]:
import os
if os.path.exists('/content/auto-browser'):
  %cd /content/auto-browser
  !git pull origin master
elif os.path.exists('auto-browser'):
  %cd auto-browser
  !git pull origin master
else:
  !git clone https://github.com/talpadeavi03/auto-browser.git
  %cd auto-browser
!ls -lh data/

## Step 3: Load Dataset & Prepare ChatML Formatting

In [ ]:
import json
from datasets import Dataset

with open("data/training_dataset.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} application Q&A interaction pairs!")

formatted_data = []
for r in records:
    text = (
        f"<|im_start|>system\n{r['instruction']}<|im_end|>\n"
        f"<|im_start|>user\n{r['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{r['output']}<|im_end|>"
    )
    formatted_data.append({"text": text})

dataset = Dataset.from_list(formatted_data)
print("Sample ChatML entry:\n", dataset[0]["text"])

## Step 4: Load Base Model with 4-bit Quantization
We use  (~1.8GB VRAM in 4-bit, fast and highly accurate).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 5: Train with SFTTrainer (Takes ~3-5 minutes on T4 GPU)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./auto_browser_lora_output",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()

## Step 6: Test Inference with the Fine-Tuned Model Live

In [ ]:
def ask_fine_tuned_agent(question: str):
    prompt = (
        f"<|im_start|>system\n{records[0]['instruction']}<|im_end|>\n"
        f"<|im_start|>user\nRecruiter Question: '{question}'\nField Type: text<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return ans.strip()

print("Test 1:", ask_fine_tuned_agent("Which AWS certification do you have?"))
print("Test 2:", ask_fine_tuned_agent("What is your notice period in days?"))
print("Test 3:", ask_fine_tuned_agent("How many years of experience do you have with Kubernetes?"))

## Step 7: Save Model for FREE
### Option A: Push to Hugging Face Hub (FREE Private/Public Repository)

In [ ]:
from huggingface_hub import login
# Paste your free Hugging Face write token
# login("hf_...")
# model.push_to_hub("talpadeavi/auto-browser-devops-agent")
# tokenizer.push_to_hub("talpadeavi/auto-browser-devops-agent")
# print("✓ Model uploaded to Hugging Face Hub!")

### Option B: Save to Google Drive (FREE)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
model.save_pretrained("/content/drive/MyDrive/auto_browser_qwen_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/auto_browser_qwen_lora")
print("✓ Model successfully saved to Google Drive!")